In [1]:
%pip install -q opencv-python sounddevice imageio-ffmpeg numpy

Note: you may need to restart the kernel to use updated packages.


In [2]:
import cv2
import sounddevice as sd

print('--- 麦克风输入设备 ---')
for i, d in enumerate(sd.query_devices()):
    if d['max_input_channels'] > 0:
        print(i, d['name'], f"({d['max_input_channels']}ch)")

print('--- 摄像头 ---')
for idx in range(2):
    cap = cv2.VideoCapture(idx)
    ok = cap.isOpened()
    if ok:
        print('摄像头', idx, '可用，分辨率', int(cap.get(3)), 'x', int(cap.get(4)), '@', int(cap.get(5)), 'fps')
    cap.release()

--- 麦克风输入设备 ---
0 Microsoft 声音映射器 - Input (2ch)
1 麦克风阵列 (适用于数字麦克风的英特尔® 智音技术) (6ch)
2 麦克风 (Steam Streaming Microphone (8ch)
3 Voicemeeter Out B1 (VB-Audio Vo (8ch)
4 Voicemeeter Out A5 (VB-Audio Vo (8ch)
5 Voicemeeter Out A2 (VB-Audio Vo (8ch)
6 Voicemeeter Out B3 (VB-Audio Vo (8ch)
7 Voicemeeter Out B2 (VB-Audio Vo (8ch)
8 Voicemeeter Out A3 (VB-Audio Vo (8ch)
9 麦克风 (Realtek(R) Audio) (2ch)
10 麦克风阵列 (网易虚拟音频设备) (2ch)
18 主声音捕获驱动程序 (2ch)
19 麦克风阵列 (适用于数字麦克风的英特尔® 智音技术) (6ch)
20 麦克风 (Steam Streaming Microphone) (8ch)
21 Voicemeeter Out B1 (VB-Audio Voicemeeter VAIO) (8ch)
22 Voicemeeter Out A5 (VB-Audio Voicemeeter VAIO) (8ch)
23 Voicemeeter Out A2 (VB-Audio Voicemeeter VAIO) (8ch)
24 Voicemeeter Out B3 (VB-Audio Voicemeeter VAIO) (8ch)
25 Voicemeeter Out B2 (VB-Audio Voicemeeter VAIO) (8ch)
26 Voicemeeter Out A3 (VB-Audio Voicemeeter VAIO) (8ch)
27 麦克风 (Realtek(R) Audio) (2ch)
28 麦克风阵列 (网易虚拟音频设备) (2ch)
42 麦克风 (Steam Streaming Microphone) (1ch)
43 Voicemeeter Out B1 (VB-Audio Voicemeeter VAI

In [3]:
import threading
import subprocess
import time
import os
import numpy as np
import cv2
import sounddevice as sd
import scipy.io.wavfile as wav
import imageio_ffmpeg

SAMPLE_RATE = 16000
FRAME_RATE = 20
VID = 'asset/_vid.avi'
AUD = 'asset/_aud.wav'
OUT = 'asset/recording.mp4'
FFMPEG = imageio_ffmpeg.get_ffmpeg_exe()


class Recorder:
    def __init__(self, cam_index=0):
        self.cam_index = cam_index
        self.on = False
        self.buf = []

    def start(self):
        self.on = True
        self.buf = []
        self.cap = cv2.VideoCapture(self.cam_index)
        w = int(self.cap.get(3))
        h = int(self.cap.get(4))
        self.vw = cv2.VideoWriter(VID, cv2.VideoWriter_fourcc(*'MJPG'), FRAME_RATE, (w, h))
        self.vt = threading.Thread(target=self._video, daemon=True)
        self.at = threading.Thread(target=self._audio, daemon=True)
        self.vt.start()
        self.at.start()

    def _video(self):
        while self.on and self.cap.isOpened():
            ret, frame = self.cap.read()
            if ret:
                self.vw.write(frame)
        self.vw.release()
        self.cap.release()

    def _audio(self):
        self.stream = sd.InputStream(
            samplerate=SAMPLE_RATE,
            channels=1,
            callback=lambda indata, n, t, s: self.buf.append(indata.copy()),
        )
        with self.stream:
            while self.on:
                sd.sleep(100)

    def stop(self):
        self.on = False
        self.vt.join(timeout=3)
        self.at.join(timeout=3)
        if self.buf:
            data = np.concatenate(self.buf, axis=0)
            wav.write(AUD, SAMPLE_RATE, (data * 32767).astype(np.int16))
        r = subprocess.run(
            [FFMPEG, '-y', '-i', VID, '-i', AUD,
             '-c:v', 'libx264', '-c:a', 'aac', '-pix_fmt', 'yuv420p', '-shortest', OUT],
            capture_output=True,
        )
        for f in (VID, AUD):
            if os.path.exists(f):
                os.remove(f)
        return r.returncode == 0, os.path.exists(OUT)

In [5]:
import tkinter as tk


class RecorderApp:
    def __init__(self, root):
        self.root = root
        root.title('音视频同步录制 (exp1D)')
        root.geometry('380x260')
        self.rec = Recorder(cam_index=0)
        self.status = tk.Label(root, text='准备就绪，点击“开始录制”')
        self.status.pack(pady=18)
        self.btn_start = tk.Button(root, text='开始录制', width=22, command=self.start)
        self.btn_start.pack(pady=5)
        self.btn_stop = tk.Button(root, text='停止并保存', width=22, command=self.stop, state='disabled')
        self.btn_stop.pack(pady=5)
        self.btn_quit = tk.Button(root, text='退出', width=22, command=root.destroy)
        self.btn_quit.pack(pady=5)

    def start(self):
        self.rec.start()
        self.status.config(text='录制中... 点击“停止并保存”结束')
        self.btn_start.config(state='disabled')
        self.btn_stop.config(state='normal')

    def stop(self):
        ok, exists = self.rec.stop()
        self.status.config(text='已保存 recording.mp4' if ok and exists else '保存失败')
        self.btn_start.config(state='normal')
        self.btn_stop.config(state='disabled')


root = tk.Tk()
app = RecorderApp(root)
root.mainloop()
print('界面已关闭')

界面已关闭
